# Computação Gráfica – Projeto 3
Adaptação do cenário do Projeto 2 para incorporar iluminação ambiente, difusa e especular.

A cena é composta por uma cabana/chalé com lareira acesa, poltrona, mesa com gramofone, uma lava lamp roxa e um gato. A área externa mantém a paisagem de neve com árvores, troncos e um vagalume.

Feito por:

* Lucas Pereira Franco de Almeida --- 12675020
* Nicolas Carreiro Rodrigues --- 14600801

Lista dos controles:

> * **[W], [A], [S], [D]:** movimentação da câmera.
> * **Mouse:** direção da câmera.
> * **Scroll:** controle do campo de visão.
> * **[ESC]:** fechar a janela.
> * **[P]:** alternar visualização da malha poligonal (para debug).
> * **[1]:** ligar/desligar luz ambiente.
> * **[2]:** ligar/desligar luz da lareira.
> * **[3]:** ligar/desligar luz da lava lamp.
> * **[4]:** ligar/desligar luz do vagalume.
> * **[5]:** ligar/desligar o movimento do vagalume.
> * **[Z]/[X]:** diminuir/aumentar luz ambiente.
> * **[C]/[V]:** diminuir/aumentar reflexão difusa.
> * **[B]/[N]:** diminuir/aumentar reflexão especular.

*Obs.: os notebooks e shaders de iluminação vistos em aula foram usados como base para este projeto.*


---
## Instalação e inicialização

### Primeiro, vamos importar as bibliotecas necessárias.

In [139]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
import ctypes
from numpy import random
from PIL import Image

from shader_s import Shader


### Inicializando janela

In [140]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 900
largura = 1600

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfwTerminate()
    
glfw.make_context_current(window)


---
## Shaders

### Constroi e compila os shaders. Também "linka" eles ao programa

#### Novidade aqui: modularização dessa parte do código --- temos agora uma classe e arquivos próprios para os shaders (vs e fs)
Créditos: https://learnopengl.com

In [141]:
ourShader = Shader("vertex_shader.vs", "fragment_shader.fs")
ourShader.use()

program = ourShader.getProgram()

---
## Carregamento de modelos e texturas

### Carregando Modelos (vértices, texturas e normais) a partir de Arquivos

A função abaixo carrega modelos a partir de arquivos no formato WaveFront (.obj).

Para saber mais sobre o modelo, acesse: https://en.wikipedia.org/wiki/Wavefront_.obj_file


In [142]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)


AMBIENTE_EXTERNO = 0
AMBIENTE_INTERNO = 1

vertices_list = []
textures_coord_list = []
normals_list = []


def load_model_from_file(filename):
    """Loads a Wavefront OBJ file. """
    vertices = []
    texture_coords = []
    normals = []
    faces = []

    material = None

    for line in open(filename, "r"):
        if line.startswith('#'): continue
        values = line.split()
        if not values: continue

        if values[0] == 'v':
            vertices.append(values[1:4])
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])
        elif values[0] == 'vn':
            normals.append(values[1:4])
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face = []
            face_texture = []
            face_normals = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                face_texture.append(int(w[1]) if len(w) >= 2 and len(w[1]) > 0 else 0)
                face_normals.append(int(w[2]) if len(w) >= 3 and len(w[2]) > 0 else 0)
            faces.append((face, face_texture, face_normals, material))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['normals'] = normals
    model['faces'] = faces

    return model


def load_texture_from_file(texture_id, img_textura):
    print(texture_id)
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    img = Image.open(img_textura).convert('RGB')
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, -1)
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)


def load_texture_from_color(texture_id, r, g, b):
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    image_data = bytes([r, g, b])
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGB, 1, 1, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)


'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
def circular_sliding_window_of_three(arr):
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result


def append_vertex(modelo, vertice_id, texture_id, normal_id):
    vertices_list.append(modelo['vertices'][vertice_id - 1])
    textures_coord_list.append(modelo['texture'][texture_id - 1] if texture_id > 0 and len(modelo['texture']) > 0 else (0, 0))
    normals_list.append(modelo['normals'][normal_id - 1] if normal_id > 0 and len(modelo['normals']) > 0 else (0, 1, 0))


def append_raw_vertex(vertex, texture_coord=(0, 0), normal=(0, 1, 0)):
    vertices_list.append(vertex)
    textures_coord_list.append(texture_coord)
    normals_list.append(normal)


numberTextures = 0


def load_obj_and_texture(objFile, texturesList):
    modelo = load_model_from_file(objFile)

    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    faces_visited = []
    for face in modelo['faces']:
        if face[3] not in faces_visited:
            faces_visited.append(face[3])
        for vertice_id, texture_id, normal_id in zip(circular_sliding_window_of_three(face[0]), circular_sliding_window_of_three(face[1]), circular_sliding_window_of_three(face[2])):
            append_vertex(modelo, vertice_id, texture_id, normal_id)

    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))

    global numberTextures
    for i in range(len(texturesList)):
        load_texture_from_file(numberTextures,texturesList[i])
        numberTextures += 1

    return verticeInicial, verticeFinal - verticeInicial


def load_obj_and_texture_fan(objFile, textureId, textureFile):
    modelo = load_model_from_file(objFile)

    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))

    for face in modelo['faces']:
        for i in range(1, len(face[0]) - 1):
            for j in [0, i, i + 1]:
                append_vertex(modelo, face[0][j], face[1][j], face[2][j])

    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))

    load_texture_from_file(textureId, textureFile)

    return verticeInicial, verticeFinal - verticeInicial


def load_obj_and_texture_xy(objFile, textureId, textureFile):
    modelo = load_model_from_file(objFile)

    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))

    x_min = min([float(v[0]) for v in modelo['vertices']])
    x_max = max([float(v[0]) for v in modelo['vertices']])
    y_min = min([float(v[1]) for v in modelo['vertices']])
    y_max = max([float(v[1]) for v in modelo['vertices']])

    for face in modelo['faces']:
        for i in range(1, len(face[0]) - 1):
            for j in [0, i, i + 1]:
                vertex = modelo['vertices'][face[0][j] - 1]
                x = float(vertex[0])
                y = float(vertex[1])
                u = (x - x_min) / (x_max - x_min)
                v = (y - y_min) / (y_max - y_min)
                append_vertex(modelo, face[0][j], 0, face[2][j])
                textures_coord_list[-1] = (u, v)

    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))

    load_texture_from_file(textureId, textureFile)

    return verticeInicial, verticeFinal - verticeInicial


def adiciona_caixa(cx, cy, cz, sx, sy, sz):
    x0, x1 = cx - sx / 2, cx + sx / 2
    y0, y1 = cy - sy / 2, cy + sy / 2
    z0, z1 = cz - sz / 2, cz + sz / 2
    faces = [
        (((x0,y0,z1),(x1,y0,z1),(x1,y1,z1),(x0,y1,z1)), (0,0,1)),
        (((x1,y0,z0),(x0,y0,z0),(x0,y1,z0),(x1,y1,z0)), (0,0,-1)),
        (((x0,y0,z0),(x0,y0,z1),(x0,y1,z1),(x0,y1,z0)), (-1,0,0)),
        (((x1,y0,z1),(x1,y0,z0),(x1,y1,z0),(x1,y1,z1)), (1,0,0)),
        (((x0,y1,z1),(x1,y1,z1),(x1,y1,z0),(x0,y1,z0)), (0,1,0)),
        (((x0,y0,z0),(x1,y0,z0),(x1,y0,z1),(x0,y0,z1)), (0,-1,0)),
    ]
    tex = [(0,0), (1,0), (1,1), (0,1)]
    for vertices_face, normal in faces:
        for a, b, c in [(0,1,2), (0,2,3)]:
            append_raw_vertex(vertices_face[a], tex[a], normal)
            append_raw_vertex(vertices_face[b], tex[b], normal)
            append_raw_vertex(vertices_face[c], tex[c], normal)


### Vamos carregar cada modelo e definir funções para desenhá-los

#### Modelos dos objetos

##### Ambiente interno

- **Gato**

In [143]:
verticeInicial_gato, quantosVertices_gato = load_obj_and_texture(
    'objetos/interno/gato/gato.obj',
    ['objetos/interno/gato/gato.jpg'] 
)

def desenha_gato(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    desenha_modelo(verticeInicial_gato, quantosVertices_gato, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_INTERNO, 0.28, 0.55, 0.12, 16, (0.0, 0.0, 0.0))


Processando modelo objetos/interno/gato/gato.obj. Vertice inicial: 0
Processando modelo objetos/interno/gato/gato.obj. Vertice final: 317592
0


- **Gramofone**

In [144]:
textureId_gramofone = glGenTextures(1)
verticeInicial_gramofone, quantosVertices_gramofone = load_obj_and_texture_fan(
    'objetos/interno/gramofone/gramofone.obj',
    textureId_gramofone,
    'objetos/interno/gramofone/gramofone.tif'
)

def desenha_gramofone(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    desenha_modelo(verticeInicial_gramofone, quantosVertices_gramofone, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_INTERNO, 0.22, 0.55, 0.45, 32, (0.0, 0.0, 0.0))

# CD ---------------------------------------------------------------------------------------

textureId_cd = glGenTextures(1)
verticeInicial_cd, quantosVertices_cd = load_obj_and_texture_fan(
    'objetos/interno/gramofone/cd.obj',
    textureId_cd,
    'objetos/interno/gramofone/cd.png'
)

def desenha_cd(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    desenha_modelo(verticeInicial_cd, quantosVertices_cd, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_INTERNO, 0.18, 0.45, 0.95, 96, (0.0, 0.0, 0.0))


Processando modelo objetos/interno/gramofone/gramofone.obj. Vertice inicial: 317592
Processando modelo objetos/interno/gramofone/gramofone.obj. Vertice final: 513849
1
Processando modelo objetos/interno/gramofone/cd.obj. Vertice inicial: 513849
Processando modelo objetos/interno/gramofone/cd.obj. Vertice final: 514617
2


- **Mesa**

In [145]:
textureId_mesa = glGenTextures(1)
verticeInicial_mesa, quantosVertices_mesa = load_obj_and_texture_fan(
    'objetos/interno/mesa/mesa.obj',
    textureId_mesa,
    'objetos/interno/mesa/mesa.png'
)

def desenha_mesa(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    desenha_modelo(verticeInicial_mesa, quantosVertices_mesa, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_INTERNO, 0.24, 0.65, 0.20, 20, (0.0, 0.0, 0.0))


Processando modelo objetos/interno/mesa/mesa.obj. Vertice inicial: 514617
Processando modelo objetos/interno/mesa/mesa.obj. Vertice final: 624705
3


- **Lava lamp**


In [146]:
textureId_lava_base = glGenTextures(1)
textureId_lava_glass = glGenTextures(1)
textureId_lava_blob = glGenTextures(1)

load_texture_from_color(textureId_lava_base, 35, 20, 45)
load_texture_from_color(textureId_lava_glass, 210, 190, 245)
load_texture_from_color(textureId_lava_blob, 230, 40, 210)


def carrega_parte_lava_lamp(objFile):
    modelo = load_model_from_file(objFile)
    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))

    for face in modelo['faces']:
        for i in range(1, len(face[0]) - 1):
            for j in [0, i, i + 1]:
                append_vertex(modelo, face[0][j], face[1][j], face[2][j])

    quantosVertices = len(vertices_list) - verticeInicial
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))
    return verticeInicial, quantosVertices


verticeInicial_lava_base, quantosVertices_lava_base = carrega_parte_lava_lamp('objetos/interno/lava lamp/lava_lamp_base.obj')
verticeInicial_lava_glass, quantosVertices_lava_glass = carrega_parte_lava_lamp('objetos/interno/lava lamp/lava_lamp_glass.obj')
verticeInicial_lava_blob, quantosVertices_lava_blob = carrega_parte_lava_lamp('objetos/interno/lava lamp/lava_lamp_blob.obj')


def desenha_lava_lamp(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    emissao_vidro = (0.02, 0.0, 0.04) if luz_lava_lamp_ligada else (0.0, 0.0, 0.0)
    emissao_core = (0.18, 0.03, 0.28) if luz_lava_lamp_ligada else (0.02, 0.0, 0.03)
    emissao_blob = (0.95, 0.08, 1.0) if luz_lava_lamp_ligada else (0.03, 0.0, 0.04)

    desenha_modelo(verticeInicial_lava_base, quantosVertices_lava_base, textureId_lava_base,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_INTERNO, 0.20, 0.50, 0.75, 80, (0.0, 0.0, 0.0))
    desenha_modelo(verticeInicial_lava_blob, quantosVertices_lava_blob, textureId_lava_blob,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_INTERNO, 0.35, 0.70, 0.65, 64, emissao_blob)
    desenha_modelo_transparente(verticeInicial_lava_glass, quantosVertices_lava_glass, textureId_lava_glass,
                                angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                                AMBIENTE_INTERNO, 0.12, 0.20, 0.95, 120, emissao_vidro, 0.18)


Processando modelo objetos/interno/lava lamp/lava_lamp_base.obj. Vertice inicial: 624705
Processando modelo objetos/interno/lava lamp/lava_lamp_base.obj. Vertice final: 631593
Processando modelo objetos/interno/lava lamp/lava_lamp_glass.obj. Vertice inicial: 631593
Processando modelo objetos/interno/lava lamp/lava_lamp_glass.obj. Vertice final: 633897
Processando modelo objetos/interno/lava lamp/lava_lamp_blob.obj. Vertice inicial: 633897
Processando modelo objetos/interno/lava lamp/lava_lamp_blob.obj. Vertice final: 639861


- **Lareira**

In [147]:
textureId_lareira_1 = glGenTextures(1)
textureId_lareira_2 = glGenTextures(1)

modelo_lareira = load_model_from_file('objetos/interno/lareira/lareira.obj')

verticeInicial_lareira_1 = len(vertices_list)
for face in modelo_lareira['faces']:
    if face[3] == 'lambert2SG':
        for i in range(1, len(face[0]) - 1):
            for j in [0, i, i + 1]:
                append_vertex(modelo_lareira, face[0][j], face[1][j], face[2][j])
quantosVertices_lareira_1 = len(vertices_list) - verticeInicial_lareira_1

verticeInicial_lareira_2 = len(vertices_list)
for face in modelo_lareira['faces']:
    if face[3] == 'lambert3SG':
        for i in range(1, len(face[0]) - 1):
            for j in [0, i, i + 1]:
                append_vertex(modelo_lareira, face[0][j], face[1][j], face[2][j])
quantosVertices_lareira_2 = len(vertices_list) - verticeInicial_lareira_2

load_texture_from_file(textureId_lareira_1, 'objetos/interno/lareira/lareira1.png')
load_texture_from_file(textureId_lareira_2, 'objetos/interno/lareira/lareira2.png')

def desenha_lareira(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId_1, textureId_2):
    desenha_modelo(verticeInicial_lareira_1, quantosVertices_lareira_1, textureId_1,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_INTERNO, 0.25, 0.75, 0.18, 18, (0.0, 0.0, 0.0))
    desenha_modelo(verticeInicial_lareira_2, quantosVertices_lareira_2, textureId_2,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_INTERNO, 0.25, 0.70, 0.22, 24, (0.0, 0.0, 0.0))


7
8


- **Poltrona**

In [148]:
textureId_poltrona = glGenTextures(1)
verticeInicial_poltrona, quantosVertices_poltrona = load_obj_and_texture_fan(
    'objetos/interno/poltrona/poltrona.obj',
    textureId_poltrona,
    'objetos/interno/poltrona/poltrona.png'
)

def desenha_poltrona(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    desenha_modelo(verticeInicial_poltrona, quantosVertices_poltrona, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_INTERNO, 0.28, 0.62, 0.12, 14, (0.0, 0.0, 0.0))


Processando modelo objetos/interno/poltrona/poltrona.obj. Vertice inicial: 641283
Processando modelo objetos/interno/poltrona/poltrona.obj. Vertice final: 643995
9


##### Ambiente externo

- **Fogueira**

In [149]:
# Fogo ---------------------------------------------------------------------------------------

textureId_fogo = glGenTextures(1)
verticeInicial_fogo, quantosVertices_fogo = load_obj_and_texture_xy(
    'objetos/externo/fogueira/fogo.obj',
    textureId_fogo,
    'objetos/externo/fogueira/fogo.jpg'
)

def desenha_fogo(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    if not luz_lareira_ligada:
        return
    emissao = (0.75, 0.28, 0.03)
    desenha_modelo(verticeInicial_fogo, quantosVertices_fogo, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_INTERNO, 0.55, 0.90, 0.18, 12, emissao)


# Madeiras ---------------------------------------------------------------------------------------

textureId_madeiras = glGenTextures(1)
verticeInicial_madeiras, quantosVertices_madeiras = load_obj_and_texture_fan(
    'objetos/externo/fogueira/madeiras.obj',
    textureId_madeiras,
    'objetos/externo/fogueira/madeira.jpg'
)

def desenha_madeiras(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    desenha_modelo(verticeInicial_madeiras, quantosVertices_madeiras, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_EXTERNO, 0.24, 0.72, 0.10, 12, (0.0, 0.0, 0.0))


Processando modelo objetos/externo/fogueira/fogo.obj. Vertice inicial: 643995
Processando modelo objetos/externo/fogueira/fogo.obj. Vertice final: 829401
10
Processando modelo objetos/externo/fogueira/madeiras.obj. Vertice inicial: 829401
Processando modelo objetos/externo/fogueira/madeiras.obj. Vertice final: 830265
11


- **Tronco**


In [150]:
textureId_tronco = glGenTextures(1)
verticeInicial_tronco, quantosVertices_tronco = load_obj_and_texture_fan(
    'objetos/externo/tronco/tronco.obj',
    textureId_tronco,
    'objetos/externo/tronco/tronco.jpg'
)

def desenha_tronco(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    desenha_modelo(verticeInicial_tronco, quantosVertices_tronco, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_EXTERNO, 0.22, 0.68, 0.10, 12, (0.0, 0.0, 0.0))


Processando modelo objetos/externo/tronco/tronco.obj. Vertice inicial: 830265
Processando modelo objetos/externo/tronco/tronco.obj. Vertice final: 991545
12


- **Tronco**

In [ ]:
textureId_arvore = glGenTextures(1)
verticeInicial_arvore, quantosVertices_arvore = load_obj_and_texture_fan(
    'objetos/externo/arvore/arvore.obj',
    textureId_arvore,
    'objetos/externo/arvore/arvore.png'
)

def desenha_arvore(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    desenha_modelo(verticeInicial_arvore, quantosVertices_arvore, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_EXTERNO, 0.25, 0.70, 0.08, 10, (0.0, 0.0, 0.0))


textureId_luz_externa = glGenTextures(1)
load_texture_from_color(textureId_luz_externa, 120, 185, 255)

verticeInicial_luz_externa = len(vertices_list)
adiciona_caixa(0.0, 0.0, 0.0, 0.35, 0.35, 0.35)
quantosVertices_luz_externa = len(vertices_list) - verticeInicial_luz_externa

def desenha_luz_externa(t_x, t_y, t_z):
    emissao = (0.15, 0.22, 0.45) if luz_externa_ligada else (0.01, 0.01, 0.03)
    desenha_modelo(verticeInicial_luz_externa, quantosVertices_luz_externa, textureId_luz_externa,
                   0, 0, 0, 0, t_x, t_y, t_z, 1, 1, 1,
                   AMBIENTE_EXTERNO, 0.20, 0.55, 0.80, 64, emissao)


Processando modelo objetos/externo/arvore/arvore.obj. Vertice inicial: 991545
Processando modelo objetos/externo/arvore/arvore.obj. Vertice final: 1106781
13


- **Vagalume**


In [152]:
textureId_vagalume = glGenTextures(1)
verticeInicial_vagalume, quantosVertices_vagalume = load_obj_and_texture_fan(
    'objetos/externo/vagalume/vagalume.obj',
    textureId_vagalume,
    'objetos/externo/vagalume/vagalume.png'
)

def desenha_vagalume(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    emissao = (0.1, 0.2, 0.0) if luz_vagalume_ligada else (0.0, 0.0, 0.0)
    desenha_modelo(verticeInicial_vagalume, quantosVertices_vagalume, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_EXTERNO, 0.3, 0.2, 0.18, 12, emissao)

Processando modelo objetos/externo/vagalume/vagalume.obj. Vertice inicial: 1106817
Processando modelo objetos/externo/vagalume/vagalume.obj. Vertice final: 1113747
15


#### Modelos do ambiente
- Skybox
- Chão externo
- Chão interno

In [153]:
# Skybox ---------------------------------------------------------------------------------------

# Os shaders específicos da skybox servem para evitar as "costuras" nas divisões das faces do cubo.
skyboxVertexShaderSource = """
#version 330 core

attribute vec3 position;
varying vec3 out_texture;

uniform mat4 model;
uniform mat4 view;
uniform mat4 projection;

void main(){
    out_texture = position;
    vec4 pos = projection * view * model * vec4(position, 1.0);
    gl_Position = pos.xyww;
}
"""

skyboxFragmentShaderSource = """
#version 330 core

varying vec3 out_texture;
uniform samplerCube skybox;

void main(){
    gl_FragColor = texture(skybox, out_texture);
}
"""

skyboxVertexShader = glCreateShader(GL_VERTEX_SHADER)
glShaderSource(skyboxVertexShader, skyboxVertexShaderSource)
glCompileShader(skyboxVertexShader)

skyboxFragmentShader = glCreateShader(GL_FRAGMENT_SHADER)
glShaderSource(skyboxFragmentShader, skyboxFragmentShaderSource)
glCompileShader(skyboxFragmentShader)

skyboxProgram = glCreateProgram()
glAttachShader(skyboxProgram, skyboxVertexShader)
glAttachShader(skyboxProgram, skyboxFragmentShader)
glLinkProgram(skyboxProgram)

glDeleteShader(skyboxVertexShader)
glDeleteShader(skyboxFragmentShader)

glUseProgram(skyboxProgram)
loc_skybox = glGetUniformLocation(skyboxProgram, "skybox")
glUniform1i(loc_skybox, 0)
ourShader.use()

verticeInicial_skybox = len(vertices_list)

skybox_vertices = [
    (-1,  1, -1), (-1, -1, -1), (1, -1, -1),
    (1, -1, -1), (1,  1, -1), (-1,  1, -1),

    (-1, -1,  1), (-1, -1, -1), (-1,  1, -1),
    (-1,  1, -1), (-1,  1,  1), (-1, -1,  1),

    (1, -1, -1), (1, -1,  1), (1,  1,  1),
    (1,  1,  1), (1,  1, -1), (1, -1, -1),

    (-1, -1,  1), (-1,  1,  1), (1,  1,  1),
    (1,  1,  1), (1, -1,  1), (-1, -1,  1),

    (-1,  1, -1), (1,  1, -1), (1,  1,  1),
    (1,  1,  1), (-1,  1,  1), (-1,  1, -1),

    (-1, -1, -1), (-1, -1,  1), (1, -1, -1),
    (1, -1, -1), (-1, -1,  1), (1, -1,  1)
]

for vertex in skybox_vertices:
    append_raw_vertex(vertex, (0, 0), (0, 1, 0))

quantosVertices_skybox = len(vertices_list) - verticeInicial_skybox

skybox_faces = [
    'objetos/ambiente/Skybox/posx.jpg',
    'objetos/ambiente/Skybox/negx.jpg',
    'objetos/ambiente/Skybox/posy.jpg',
    'objetos/ambiente/Skybox/negy.jpg',
    'objetos/ambiente/Skybox/posz.jpg',
    'objetos/ambiente/Skybox/negz.jpg'
]

textureId_skybox = glGenTextures(1)
glBindTexture(GL_TEXTURE_CUBE_MAP, textureId_skybox)

for i in range(len(skybox_faces)):
    img = Image.open(skybox_faces[i])
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGB", 0, 1)
    glTexImage2D(GL_TEXTURE_CUBE_MAP_POSITIVE_X + i, 0, GL_RGB, img_width, img_height, 0, GL_RGB, GL_UNSIGNED_BYTE, image_data)

glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)
glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_R, GL_CLAMP_TO_EDGE)

def desenha_skybox(t_x, t_y, t_z, textureId):

    glDepthFunc(GL_LEQUAL)
    glUseProgram(skyboxProgram)

    mat_model = model(0, 0, 0, 0, t_x, t_y, t_z, 50, 50, 50)
    loc_model = glGetUniformLocation(skyboxProgram, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    mat_view = view()
    loc_view = glGetUniformLocation(skyboxProgram, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(skyboxProgram, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)

    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
    stride = vertices.strides[0]
    offset = ctypes.c_void_p(0)
    loc_vertices_skybox = glGetAttribLocation(skyboxProgram, "position")
    glEnableVertexAttribArray(loc_vertices_skybox)
    glVertexAttribPointer(loc_vertices_skybox, 3, GL_FLOAT, False, stride, offset)

    glBindTexture(GL_TEXTURE_CUBE_MAP, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial_skybox, quantosVertices_skybox)

    glDepthFunc(GL_LESS)
    ourShader.use()


# Chao externo ---------------------------------------------------------------------------------------

verticeInicial_chao_ext = len(vertices_list)
textureId_chao_ext = glGenTextures(1)

vertices_chao_ext = [
    (-300, -0.5, -300), (300, -0.5, -300), (300, -0.5, 300),
    (-300, -0.5, -300), (300, -0.5, 300), (-300, -0.5, 300)
]

textures_chao_ext = [
    (0, 0), (60, 0), (60, 60),
    (0, 0), (60, 60), (0, 60)
]

for vertex, texture_coord in zip(vertices_chao_ext, textures_chao_ext):
    append_raw_vertex(vertex, texture_coord, (0, 1, 0))

load_texture_from_file(textureId_chao_ext, 'objetos/ambiente/Chao_ext/chao_neve.jpg')

quantosVertices_chao_ext = len(vertices_list) - verticeInicial_chao_ext

def desenha_chao_ext(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    desenha_modelo(verticeInicial_chao_ext, quantosVertices_chao_ext, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_EXTERNO, 0.30, 0.82, 0.35, 32, (0.0, 0.0, 0.0))


# Cabana ---------------------------------------------------------------------------------------

textureId_cabana = glGenTextures(1)
verticeInicial_cabana, quantosVertices_cabana = load_obj_and_texture_fan(
    'objetos/ambiente/Cabana/Snow covered CottageOBJ.obj',
    textureId_cabana,
    'objetos/ambiente/Cabana/Cottage Texture.jpg'
)

def desenha_cabana(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId, ambiente):
    desenha_modelo(verticeInicial_cabana, quantosVertices_cabana, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   ambiente, 0.25, 0.75, 0.22, 20, (0.0, 0.0, 0.0))


# Chao interno ---------------------------------------------------------------------------------------

verticeInicial_chao_int = len(vertices_list)
textureId_chao_int = glGenTextures(1)

vertices_chao_int = [
    (-28, -2.35, -58), (28, -2.35, -58), (28, -2.35, 38),
    (-28, -2.35, -58), (28, -2.35, 38), (-28, -2.35, 38)
]

textures_chao_int = [
    (0, 0), (6, 0), (6, 10),
    (0, 0), (6, 10), (0, 10)
]

for vertex, texture_coord in zip(vertices_chao_int, textures_chao_int):
    append_raw_vertex(vertex, texture_coord, (0, 1, 0))

load_texture_from_file(textureId_chao_int, 'objetos/ambiente/Chao_int/chao_madeira.jpg')

quantosVertices_chao_int = len(vertices_list) - verticeInicial_chao_int

def desenha_chao_int(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, textureId):
    desenha_modelo(verticeInicial_chao_int, quantosVertices_chao_int, textureId,
                   angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z,
                   AMBIENTE_INTERNO, 0.26, 0.72, 0.20, 18, (0.0, 0.0, 0.0))

17
Processando modelo objetos/ambiente/Cabana/Snow covered CottageOBJ.obj. Vertice inicial: 1113789
Processando modelo objetos/ambiente/Cabana/Snow covered CottageOBJ.obj. Vertice final: 1115571
18
19


---
## Envio ao buffer e alocação na GPU

### Para enviar nossos dados da CPU para a GPU, precisamos requisitar três slots (buffers): um para os vértices, um para as texturas e outro para as normais.


In [154]:
buffer_VBO = glGenBuffers(3)


### Enviando coordenadas de vértices para a GPU

Veja os parâmetros da função glBufferData [https://www.khronos.org/registry/OpenGL-Refpages/gl4/html/glBufferData.xhtml]

In [155]:
vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
vertices['position'] = vertices_list


# Upload data
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
stride = vertices.strides[0]
offset = ctypes.c_void_p(0)
loc_vertices = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_vertices)
glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

### Enviando coordenadas de textura para a GPU

In [156]:
textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)])
textures['position'] = textures_coord_list


glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
stride = textures.strides[0]
offset = ctypes.c_void_p(0)
loc_texture_coord = glGetAttribLocation(program, "texture_coord")

glEnableVertexAttribArray(loc_texture_coord)
glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)

normals = np.zeros(len(normals_list), [("position", np.float32, 3)])
normals['position'] = normals_list


glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[2])
glBufferData(GL_ARRAY_BUFFER, normals.nbytes, normals, GL_STATIC_DRAW)
stride = normals.strides[0]
offset = ctypes.c_void_p(0)
loc_normals_coord = glGetAttribLocation(program, "normals")

glEnableVertexAttribArray(loc_normals_coord)
glVertexAttribPointer(loc_normals_coord, 3, GL_FLOAT, False, stride, offset)


---
## Eventos de teclado e câmera

### Eventos de teclado

**Posição da câmera:**
* Teclas A, S, D e W para movimentação no espaço tridimensional
* Posição do mouse para "direcionar" a câmera

**Demais eventos:**
* **[P]:** alternar visualização da malha poligonal (usamos para fazer alguns debugs).

**Luzes e materiais:**
* **[1]:** ligar/desligar luz ambiente.
* **[2]:** ligar/desligar luz da lareira.
* **[3]:** ligar/desligar luz da lava lamp.
* **[4]:** ligar/desligar luz do vagalume.
* **[5]:** ligar/desligar o movimento do vagalume.
* **[Z]/[X]:** diminuir/aumentar luz ambiente.
* **[C]/[V]:** diminuir/aumentar reflexão difusa.
* **[B]/[N]:** diminuir/aumentar reflexão especular.


In [157]:
'''
Variáveis globais dos eventos de teclado e iluminação
'''

cameraPos   = glm.vec3(-0.279745, 0.800512, 3.0662)
cameraFront = glm.vec3(0.0, 0.0, -1.0)
cameraUp    = glm.vec3(0.0, 1.0, 0.0)

firstMouse = True
yaw = -90.0
pitch = 0.0
lastX = largura / 2.0
lastY = altura / 2.0
fov = 45.0

deltaTime = 0.0
lastFrame = 0.0

limite_camera_x_min = -20.0
limite_camera_x_max = 20.0
limite_camera_y_min = 0.0
limite_camera_y_max = 10.0
limite_camera_z_min = -20.0
limite_camera_z_max = 20.0

luz_ambiente_ligada = True
luz_lareira_ligada = True
luz_lava_lamp_ligada = True
luz_vagalume_ligada = True
vagalume_anda = True

tempo_vagalume = 0.0
angulo_vagalume = 0.0
velocidade_giro_vagalume = 57.0

intensidade_ambiente = 1
difusa_global = 1.0
especular_global = 3.0
passo_luz = 0.05
polygonal_mode = False

escala_fogo_min = 0.07
escala_fogo_max = 0.18
escala_fogo_1 = escala_fogo_min
escala_fogo_2 = escala_fogo_min
escala_fogo_metade = escala_fogo_min + (escala_fogo_max - escala_fogo_min) / 2
fogo_2_ativo = False
velocidade_escala_fogo = 0.10
duracao_ciclo_fogo = (escala_fogo_max - escala_fogo_min) / velocidade_escala_fogo
tempo_fogo = 0.0

angulo_cd = 0.0
velocidade_cd = 45.0

lightPosLareira = glm.vec3(-8.05, 0.05, -10.25)
lightPosLavaLamp = glm.vec3(-6.6, 0.62, -13.1)
lightPosVagalume = glm.vec3(11.5, 0.2, -8.5)


In [158]:
'''
Definição dos eventos de teclado
'''

def limita(valor, minimo, maximo):
    return max(minimo, min(valor, maximo))


def key_event(window,key,scancode,action,mods):
    global cameraPos, cameraFront, cameraUp
    global luz_ambiente_ligada, luz_lareira_ligada, luz_lava_lamp_ligada, luz_vagalume_ligada
    global vagalume_anda
    global intensidade_ambiente, difusa_global, especular_global
    global polygonal_mode

    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)

    cameraSpeed = 20 * deltaTime
    if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
    if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    cameraPos.x = max(limite_camera_x_min, min(cameraPos.x, limite_camera_x_max))
    cameraPos.y = max(limite_camera_y_min, min(cameraPos.y, limite_camera_y_max))
    cameraPos.z = max(limite_camera_z_min, min(cameraPos.z, limite_camera_z_max))

    if action == glfw.PRESS:
        if key == glfw.KEY_P:
            polygonal_mode = not polygonal_mode
        if key == glfw.KEY_1:
            luz_ambiente_ligada = not luz_ambiente_ligada
        if key == glfw.KEY_2:
            luz_lareira_ligada = not luz_lareira_ligada
        if key == glfw.KEY_3:
            luz_lava_lamp_ligada = not luz_lava_lamp_ligada
        if key == glfw.KEY_4:
            luz_vagalume_ligada = not luz_vagalume_ligada
        if key == glfw.KEY_5:
            vagalume_anda = not vagalume_anda


    if action == glfw.PRESS or action == glfw.REPEAT:
        if key == glfw.KEY_Z:
            intensidade_ambiente = limita(intensidade_ambiente - passo_luz, 0.0, 1.0)
        if key == glfw.KEY_X:
            intensidade_ambiente = limita(intensidade_ambiente + passo_luz, 0.0, 1.0)
        if key == glfw.KEY_C:
            difusa_global = limita(difusa_global - passo_luz, 0.0, 2.0)
        if key == glfw.KEY_V:
            difusa_global = limita(difusa_global + passo_luz, 0.0, 2.0)
        if key == glfw.KEY_B:
            especular_global = limita(especular_global - passo_luz, 0.0, 2.0)
        if key == glfw.KEY_N:
            especular_global = limita(especular_global + passo_luz, 0.0, 2.0)


def framebuffer_size_callback(window, largura, altura):
    glViewport(0, 0, largura, altura)


def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch

    if (firstMouse):
        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)


def scroll_callback(window, xoffset, yoffset):
    global fov
    fov -= yoffset
    if (fov < 1.0):
        fov = 1.0
    if (fov > 45.0):
        fov = 45.0


glfw.set_key_callback(window,key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)


---
## Matrizes de transformação

### Matrizes Model, View e Projection

In [159]:
def model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    angle = math.radians(angle)
    matrix_transform = glm.mat4(1.0)
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))
    if angle!=0:
        matrix_transform = glm.rotate(matrix_transform, angle, glm.vec3(r_x, r_y, r_z))
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))
    matrix_transform = np.array(matrix_transform)
    return matrix_transform


def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp);
    mat_view = np.array(mat_view)
    return mat_view


def projection():
    global altura, largura
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 500.0)
    mat_projection = np.array(mat_projection)
    return mat_projection


def envia_matriz_modelo(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    mat_model = model(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)


def envia_material(ambiente, ka, kd, ks, ns, emissive, opacity=1.0):
    glUniform1i(glGetUniformLocation(program, "ambienteObjeto"), ambiente)
    glUniform1f(glGetUniformLocation(program, "ka"), ka)
    glUniform1f(glGetUniformLocation(program, "kd"), kd)
    glUniform1f(glGetUniformLocation(program, "ks"), ks)
    glUniform1f(glGetUniformLocation(program, "ns"), ns)
    glUniform1f(glGetUniformLocation(program, "opacity"), opacity)
    glUniform3f(glGetUniformLocation(program, "emissiveColor"), emissive[0], emissive[1], emissive[2])


def desenha_modelo(verticeInicial, quantosVertices, textureId, angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, ambiente, ka, kd, ks, ns, emissive):
    envia_matriz_modelo(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    envia_material(ambiente, ka, kd, ks, ns, emissive)
    glBindTexture(GL_TEXTURE_2D, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial, quantosVertices)


def desenha_modelo_transparente(verticeInicial, quantosVertices, textureId, angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z, ambiente, ka, kd, ks, ns, emissive, opacity):
    envia_matriz_modelo(angle, r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z)
    envia_material(ambiente, ka, kd, ks, ns, emissive, opacity)
    glDepthMask(GL_FALSE)
    glBindTexture(GL_TEXTURE_2D, textureId)
    glDrawArrays(GL_TRIANGLES, verticeInicial, quantosVertices)
    glDepthMask(GL_TRUE)
    glUniform1f(glGetUniformLocation(program, "opacity"), 1.0)


def envia_luzes():
    glUniform1i(glGetUniformLocation(program, "imagem"), 0)
    glUniform3f(glGetUniformLocation(program, "viewPos"), cameraPos.x, cameraPos.y, cameraPos.z)

    glUniform1i(glGetUniformLocation(program, "luzAmbienteLigada"), int(luz_ambiente_ligada))
    glUniform1i(glGetUniformLocation(program, "luzLareiraLigada"), int(luz_lareira_ligada))
    glUniform1i(glGetUniformLocation(program, "luzLavaLampLigada"), int(luz_lava_lamp_ligada))
    glUniform1i(glGetUniformLocation(program, "luzVagalumeLigada"), int(luz_vagalume_ligada))

    glUniform1f(glGetUniformLocation(program, "intensidadeAmbiente"), intensidade_ambiente)
    glUniform1f(glGetUniformLocation(program, "difusaGlobal"), difusa_global)
    glUniform1f(glGetUniformLocation(program, "especularGlobal"), especular_global)

    glUniform3f(glGetUniformLocation(program, "lightPosLareira"), lightPosLareira.x, lightPosLareira.y, lightPosLareira.z)
    glUniform3f(glGetUniformLocation(program, "lightPosLavaLamp"), lightPosLavaLamp.x, lightPosLavaLamp.y, lightPosLavaLamp.z)
    glUniform3f(glGetUniformLocation(program, "lightPosVagalume"), lightPosVagalume.x, lightPosVagalume.y, lightPosVagalume.z)

    glUniform3f(glGetUniformLocation(program, "lightColorLareira"), 1.0, 0.45, 0.08)
    glUniform3f(glGetUniformLocation(program, "lightColorLavaLamp"), 0.72, 0.15, 1.0)
    glUniform3f(glGetUniformLocation(program, "lightColorVagalume"), 0.2, 0.4, 0.0)


---
## Janela final

### Nesse momento, nós exibimos a janela!


In [160]:
glfw.maximize_window(window)
glfw.show_window(window)

### Loop principal da janela.

In [ ]:
glEnable(GL_DEPTH_TEST)

while not glfw.window_should_close(window):

    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    angulo_cd += velocidade_cd * deltaTime

    
    fogo_x = -8.05
    fogo_y = -0.3
    fogo_z = -10.25

    
    if vagalume_anda:
        tempo_vagalume += deltaTime 
        angulo_vagalume += velocidade_giro_vagalume * deltaTime

    lightPosVagalume.x = 1.5 + math.sin(tempo_vagalume) * 2.5
    lightPosVagalume.z = -4.5 + math.cos(tempo_vagalume) * 2.5

    lightPosLareira.x = fogo_x - 0.197 * escala_fogo_1
    lightPosLareira.y = fogo_y + 3.508 * escala_fogo_1
    lightPosLareira.z = fogo_z + 0.148 * escala_fogo_1
    lightPosLavaLamp.x = -6.6
    lightPosLavaLamp.y = 0.45 + 2.75 * 0.06
    lightPosLavaLamp.z = -13.1

    glfw.poll_events()

    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(1.0, 1.0, 1.0, 1.0)

    if polygonal_mode:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

    desenha_skybox(cameraPos.x, cameraPos.y, cameraPos.z, textureId_skybox)

    ourShader.use()

    mat_view = view()
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)

    mat_projection = projection()
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)

    envia_luzes()

    desenha_chao_ext(0, 0, 0, 0, 0, 0, 0, 1, 1, 1, textureId_chao_ext)
    desenha_cabana(225, 0, 1, 0, -7, -0.5, -12, 0.08, 0.08, 0.08, textureId_cabana, AMBIENTE_EXTERNO)
    desenha_chao_int(225, 0, 1, 0, -7.5, 0.0001, -12.5, 0.079, 0.08, 0.075, textureId_chao_int)
    desenha_gramofone(0, 0, 1, 0, -6.75, 0.42, -13.55, 0.012, 0.012, 0.012, textureId_gramofone)
    desenha_cd(angulo_cd, 0, 1, 0, -6.75 , 0.698, -13.55 , 0.2, 0.2, 0.2, textureId_cd)
    desenha_mesa(0, 0, 1, 0, -6.75, -0.15, -13.35, 0.4, 0.4, 0.4, textureId_mesa)
    desenha_lava_lamp(225, 0, 1, 0, -6.6, 0.45, -13.1, 0.06, 0.06, 0.06)
    desenha_lareira(-225, 0, 1, 0, -7.5, -0.15, -9.9, 0.01, 0.01, 0.01, textureId_lareira_1, textureId_lareira_2)
    desenha_poltrona(315, 0, 1, 0, -5.7, -0.25, -12.5, 0.008, 0.008, 0.008, textureId_poltrona)
    desenha_gato(-90, 1, 0, 0, -7.5, -0.2, -12, 0.015, 0.015, 0.015, 0)
    desenha_tronco(-90, 1, 0, 0, -0.2, -0.4, 2, 0.12, 0.045, 0.045, textureId_tronco)
    desenha_tronco(-90, 1, 0, 0, -0.2, -0.4, -2, 0.12, 0.045, 0.045, textureId_tronco)
    desenha_arvore(0, 0, 0, 0, 0, -1, -20, 7, 7, 7, textureId_arvore)
    desenha_arvore(18, 0, 1, 0, -14, -1, -23, 5.8, 5.8, 5.8, textureId_arvore)
    desenha_arvore(-12, 0, 1, 0, 11, -1, -18, 6.4, 6.4, 6.4, textureId_arvore)
    desenha_arvore(34, 0, 1, 0, -18, -1, -6, 4.9, 4.9, 4.9, textureId_arvore)
    desenha_arvore(-27, 0, 1, 0, 15, -1, 5, 5.4, 5.4, 5.4, textureId_arvore)
    desenha_arvore(9, 0, 1, 0, -9, -1, 10, 4.6, 4.6, 4.6, textureId_arvore)
    desenha_arvore(41, 0, 1, 0, 22, -1, -8, 6.9, 6.9, 6.9, textureId_arvore)
    desenha_arvore(-38, 0, 1, 0, -24, -1, 14, 5.2, 5.2, 5.2, textureId_arvore)
    desenha_madeiras(90, 0, 1, 0, -0.05, -0.1, -0.05, 0.2, 0.2, 0.2, textureId_madeiras)
    desenha_vagalume(angulo_vagalume, 0, 1, 0, lightPosVagalume.x, lightPosVagalume.y, lightPosVagalume.z, 1.0, 1.0, 1.0, textureId_vagalume)


    tempo_fogo += deltaTime
    progresso_fogo_1 = (tempo_fogo % duracao_ciclo_fogo) / duracao_ciclo_fogo
    escala_fogo_1 = escala_fogo_min + progresso_fogo_1 * (escala_fogo_max - escala_fogo_min)

    if tempo_fogo >= duracao_ciclo_fogo / 2:
        fogo_2_ativo = True
        progresso_fogo_2 = ((tempo_fogo - duracao_ciclo_fogo / 2) % duracao_ciclo_fogo) / duracao_ciclo_fogo
        escala_fogo_2 = escala_fogo_min + progresso_fogo_2 * (escala_fogo_max - escala_fogo_min)

    desenha_fogo(0, 0, 0, 0, fogo_x, fogo_y, fogo_z, escala_fogo_1, escala_fogo_1, escala_fogo_1, textureId_fogo)
    if fogo_2_ativo:
        desenha_fogo(0, 0, 0, 0, fogo_x-0.1, fogo_y, fogo_z, escala_fogo_2, escala_fogo_2, escala_fogo_2, textureId_fogo)

    glfw.swap_buffers(window)

glfw.terminate()
